# Extract MFCC Mean Feature Datasets

This notebook creates train, validation and test datasets based on MFCC mean features.

For each selected track, several random audio segments are extracted. For each segment, MFCC-based summary features are calculated. The resulting tabular datasets are used for classical machine learning models.

In [ ]:
import sys
from pathlib import Path

current_path = Path.cwd()

for parent in [current_path] + list(current_path.parents):
    if (parent / "src").exists():
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError("Could not find project root containing 'src' folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import os
import glob
import pandas as pd
from tqdm import tqdm

from src.config import (
    PROCESSED_TRACKS_CSV,
    WAV_AUDIO_DIR,
    SPLIT_OUTPUT_DIR,
    SAMPLE_RATE,
    N_MFCC,
    SEGMENT_LENGTH_SECONDS,
    N_SEGMENTS_PER_TRACK,
    N_TRAIN_PER_GENRE,
    N_VAL_PER_GENRE,
    N_TEST_PER_GENRE,
    EXCLUDED_GENRES,
    USE_TOP_GENRES,
    TOP_N_GENRES
)

from src.preprocessing_functions import (
    fixed_split_per_genre_grouped,
    extract_mfcc_random_mean_df
)

In [ ]:
base_dir = str(WAV_AUDIO_DIR)
output_dir = str(SPLIT_OUTPUT_DIR)

SAMPLE_RATE = SAMPLE_RATE
N_MFCC = N_MFCC
LENGTH_IN_SECONDS = SEGMENT_LENGTH_SECONDS
N_SAMPLES = N_SEGMENTS_PER_TRACK

print("Base directory:", base_dir)
print("Output directory:", output_dir)
print("MFCCs:", N_MFCC)
print("Segment length:", LENGTH_IN_SECONDS)
print("Samples per track:", N_SAMPLES)

In [ ]:
# Load metadata containing track IDs and genre labels
metadata = pd.read_csv(PROCESSED_TRACKS_CSV)[["track_id", "track_genre_top"]]

# List all available WAV files in the dataset directory
available_files = [f for f in os.listdir(base_dir) if f.endswith(".wav")]
print(len(available_files), "WAV-Dateien im Verzeichnis gefunden.")

# Searches for _aug which marks augmented files, sets separator accordingly
if any("_aug" in f for f in available_files):
    sep = "_"
else:
    sep = "."

print(f"Used separator: '{sep}'")

# Extracts track IDs from available files
available_track_ids = set()

for f in available_files:
    try:
        # Extract track ID from filename
        track_id = str(int(f.split(sep)[0]))
        available_track_ids.add(track_id)

    except ValueError:
        # Handle files that do not follow the expected naming format
        print(f"Track_ID could not be extracted {f}")

# Filter metadata to only include available track_ids
metadata_filtered = metadata[
    metadata["track_id"].astype(str).isin(available_track_ids)
]

# Print dataset statistics
print(f"Metadata before: {len(metadata)}")
print(f"Metadata after filter on available track_IDs: {len(metadata_filtered)}")

In [ ]:
print(metadata_filtered["track_genre_top"].value_counts(dropna=False))

In [ ]:
splits, split_files = fixed_split_per_genre_grouped(
    metadata_filtered,
    base_dir,
    n_train=N_TRAIN_PER_GENRE,
    n_val=N_VAL_PER_GENRE,
    n_test=N_TEST_PER_GENRE,
    separator=sep,
    exclude_genres=EXCLUDED_GENRES,
    use_top_genres=USE_TOP_GENRES,
    top_n=TOP_N_GENRES
)

In [ ]:
# Define base directories for dataset splits
base_dirs = {
    "train": base_dir,
    "val": base_dir,
    "test": base_dir
}

# Ensure that the output directory for the extracted features exists
os.makedirs(output_dir, exist_ok=True)

# Iterate over dataset splits
for split_name, split_df in splits.items():
    print(f"\n[INFO] Processing split: {split_name} ({len(split_df)} Tracks)")

    current_base_dir = base_dirs[split_name]

    data = []

    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"{split_name}"):
        track_id = str(row["track_id"])
        genre = row["track_genre_top"]

        # Find all WAV files belonging to this track in the directory of the current split
        pattern = os.path.join(current_base_dir, f"{track_id}*.wav")
        file_list = sorted(glob.glob(pattern))

        if not file_list:
            print(f"[WARN] No files found for Track {track_id} in {current_base_dir}")
            continue

        for file_path in file_list:
            file_name = os.path.basename(file_path)

            mfcc_df = extract_mfcc_random_mean_df(
                file_path=file_path,
                track_id=track_id,
                length_in_seconds=LENGTH_IN_SECONDS,
                n_samples=N_SAMPLES,
                n_mfcc=N_MFCC
            )

            if not mfcc_df.empty:
                mfcc_df["split"] = split_name
                mfcc_df["genre"] = genre
                mfcc_df["file_name"] = file_name
                mfcc_df["is_augmented"] = "_aug" in file_name.lower()
                data.append(mfcc_df)

    if data:
        df_mfcc = pd.concat(data, ignore_index=True)

        out_path = os.path.join(
            output_dir,
            f"{split_name}_large_mfcc_mean_{LENGTH_IN_SECONDS}s_{N_SAMPLES}_427_53_53_filtered_genres.pkl"
        )

        df_mfcc.to_pickle(out_path)
        print(f"[SAVED] {split_name}: {len(df_mfcc)} Segmente → {out_path}")
    else:
        print(f"[WARN] No features calculated for Split {split_name}!")